# E5: Feature Selection Funnel — L1 (Univariate Predictive Power)

**Spec sections**: 6.2 (all), 6.8, 7.3  
**Input**: DEV window, `sample_status ∈ {GOOD, BAD}`; features with `L0_pass = Y`; `y = (sample_status == "BAD")`.

In [ ]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" --quiet
dbutils.library.restartPython()

In [ ]:
import sys, os, hashlib, shutil, tempfile, json, ast, time, pickle, warnings
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from optbinning import OptimalBinning
import common

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
BINNING_DIR = os.path.join(ARTIFACTS, "binning", "L1")
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(BINNING_DIR, exist_ok=True)

cat = cfg["databricks"]["catalog"]
sch = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L1 = cfg["funnel"]["L1"]
BIN_CFG = L1["binning"]
DEV_START, DEV_END = cfg["windows"]["DEV"]
target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]
SEED = cfg["random_seed"]

# Funnel config hash check (AGENTS.md: E5–E9 recompute and stop if changed)
funnel_config_hash = hashlib.sha256(
    json.dumps(cfg["funnel"], sort_keys=True).encode()
).hexdigest()

# Read the L0 run summary to get the frozen hash
run_summary_path = os.path.join(REPORTS, "run_summaries", "s03_L0.json")
assert os.path.exists(run_summary_path), f"L0 run summary not found: {run_summary_path}"
with open(run_summary_path) as f:
    l0_summary = json.load(f)
l0_funnel_hash = l0_summary["funnel_config_hash"]

# Check for threshold_changes.csv
tc_path = os.path.join(PROJECT_ROOT, "config", "threshold_changes.csv")
if funnel_config_hash != l0_funnel_hash:
    if os.path.exists(tc_path):
        tc = pd.read_csv(tc_path)
        print(f"\u26a0 Funnel config hash changed but threshold_changes.csv has {len(tc)} row(s) — proceeding.")
        print(tc.to_string())
    else:
        raise ValueError(
            f"Funnel config hash mismatch!\n"
            f"  L0 frozen : {l0_funnel_hash[:12]}\n"
            f"  Current   : {funnel_config_hash[:12]}\n"
            f"Add a row to config/threshold_changes.csv explaining the change."
        )
else:
    print(f"\u2705 Funnel config hash matches L0: {funnel_config_hash[:12]}")

_TMP = tempfile.mkdtemp(prefix="e5_")
_t0 = time.time()

print(f"Config hash       : {common.config_hash(cfg)[:12]}")
print(f"Master table      : {MASTER_TBL}")
print(f"DEV window        : [{DEV_START}, {DEV_END})")
print(f"L1 binning        : {BIN_CFG}")
print(f"IV thresholds     : iv_min={L1['iv_min']}, review_band={L1['iv_review_band']}, "
      f"leak_review={L1['iv_leak_review']}, cv_folds={L1['cv_iv_folds']}")

In [ ]:
# ── Load master table and feature log ───────────────────────────────────────
master = spark.table(MASTER_TBL).toPandas()
dev = master[master["window"] == "DEV"].copy()

# L1 input: DEV GOOD + BAD only (spec 6.2)
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])].copy()
common.assert_labelled(dev_gb, cfg)
y = (dev_gb[status_col] == "BAD").astype(int)
print(f"DEV GOOD+BAD rows : {len(dev_gb):,}  |  Bads: {y.sum()}")

# Feature log from L0
flog = spark.table(FLOG_TBL).toPandas()
print(f"Feature log rows  : {len(flog)}")

# L0 survivors
l0_pass = flog[flog["L0_pass"] == "Y"]["feature"].tolist()
l0_pass_in_data = [f for f in l0_pass if f in dev_gb.columns]
print(f"L0 survivors      : {len(l0_pass)} (in data: {len(l0_pass_in_data)})")

# Feature dictionary for special_codes and expected_direction
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
fdict_idx = fdict.set_index("feature")

# Stratification key for CV (spec 5.5.1)
strat_key = dev_gb[target_col].astype(str) + "_" + dev_gb["app_quarter"].astype(str)

In [ ]:
# ── L1: Fit OptimalBinning per feature (spec 6.2.1) ─────────────────

def parse_special_codes(sc_str):
    """Parse special_codes from feature_dict."""
    sc = str(sc_str).strip()
    if not sc or sc == "nan":
        return None
    try:
        p = ast.literal_eval(sc)
        return list(p) if isinstance(p, (list, tuple)) else None
    except Exception:
        return None

def fit_one_feature(feat, x, y_arr, dtype, special_codes, expected_dir):
    """Fit OptimalBinning and return (binning_object, record_dict)."""
    rec = {"feature": feat, "iv_dev": np.nan, "n_bins": np.nan,
           "trend": None, "solver_status": None}

    is_cat = (dtype == "categorical")
    mono = BIN_CFG["monotonic_trend"]

    # expected_direction = 0: try peak and valley, keep higher IV
    if str(expected_dir).strip() == "0":
        trends_to_try = ["peak", "valley"]
    else:
        trends_to_try = [mono]

    best_ob, best_iv, best_trend = None, -1, None
    for tr in trends_to_try:
        kwargs = dict(
            name=feat,
            dtype="categorical" if is_cat else "numerical",
            solver=BIN_CFG["solver"],
            monotonic_trend=tr if not is_cat else None,
            min_bin_size=BIN_CFG["min_bin_size"],
            min_bin_n_event=BIN_CFG["min_bin_n_event"],
            max_n_prebins=BIN_CFG["max_n_prebins"],
            max_n_bins=BIN_CFG["max_n_bins"],
        )
        if is_cat:
            kwargs["cat_cutoff"] = BIN_CFG["cat_cutoff"]
        if special_codes is not None:
            kwargs["special_codes"] = special_codes

        ob = OptimalBinning(**kwargs)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                ob.fit(x, y_arr)
            except (ValueError, TypeError):
                # Categorical cutoff may remove all categories; retry without it
                if is_cat and "cat_cutoff" in kwargs:
                    kw2 = {k: v for k, v in kwargs.items() if k != "cat_cutoff"}
                    ob = OptimalBinning(**kw2)
                    try:
                        ob.fit(x, y_arr)
                    except Exception:
                        continue
                else:
                    continue

        tbl = ob.binning_table.build()
        # IV is the last row, "Totals"
        iv = tbl.loc[tbl.index[-1], "IV"] if "IV" in tbl.columns else 0.0
        if np.isnan(iv):
            iv = 0.0
        if iv > best_iv:
            best_iv = iv
            best_ob = ob
            best_trend = tr

    rec["iv_dev"] = round(best_iv, 6)
    rec["solver_status"] = best_ob.status if best_ob else None

    # Determine actual trend from the fitted binning table
    if best_ob is not None:
        tbl = best_ob.binning_table.build()
        # Non-special, non-missing, non-totals rows
        core = tbl[~tbl.index.isin(["Special", "Missing", "Totals", tbl.index[-1]])].copy()
        if "WoE" in core.columns and len(core) >= 2:
            woe_vals = pd.to_numeric(core["WoE"], errors="coerce").dropna().values
            if len(woe_vals) < 2:
                rec["trend"] = best_trend
            elif all(woe_vals[i] <= woe_vals[i+1] for i in range(len(woe_vals)-1)):
                rec["trend"] = "ascending"
            elif all(woe_vals[i] >= woe_vals[i+1] for i in range(len(woe_vals)-1)):
                rec["trend"] = "descending"
            else:
                # Check peak / valley
                diffs = np.diff(woe_vals)
                sign_changes = np.where(np.diff(np.sign(diffs)))[0]
                if len(sign_changes) == 1:
                    if diffs[0] > 0:
                        rec["trend"] = "peak"
                    else:
                        rec["trend"] = "valley"
                else:
                    rec["trend"] = "non_monotonic"
        else:
            rec["trend"] = best_trend
        # Count regular bins (excluding Special, Missing, Totals)
        rec["n_bins"] = len(core)
    else:
        rec["trend"] = best_trend

    if str(expected_dir).strip() == "0":
        rec["trend_note"] = "U-shape per dictionary"
    else:
        rec["trend_note"] = None

    return best_ob, rec

# ── Run binning ───────────────────────────────────────────────────────
records = []
binning_objects = {}  # feat -> OptimalBinning
binning_tables = {}   # feat -> DataFrame

for i, feat in enumerate(l0_pass_in_data):
    info = fdict_idx.loc[feat]
    dtype = str(info.get("dtype", "numerical")).strip()
    sc = parse_special_codes(info.get("special_codes", ""))
    ed = str(info.get("expected_direction", "?")).strip()

    x = dev_gb[feat].copy()
    # For categorical: ensure string type; for numerical: ensure numeric
    if dtype == "categorical":
        x = x.astype(str).replace("nan", np.nan)
    else:
        x = pd.to_numeric(x, errors="coerce")

    ob, rec = fit_one_feature(feat, x, y.values, dtype, sc, ed)
    records.append(rec)
    if ob is not None:
        binning_objects[feat] = ob
        binning_tables[feat] = ob.binning_table.build()

    if (i + 1) % 50 == 0:
        print(f"  Fitted {i+1}/{len(l0_pass_in_data)} features...")

l1_df = pd.DataFrame(records)
print(f"\nBinning complete: {len(l1_df)} features")
print(f"IV summary:\n{l1_df['iv_dev'].describe().round(4)}")
print(f"\nSolver status:\n{l1_df['solver_status'].value_counts()}")

In [ ]:
# ── Direction conflict check (spec 6.2.1) ──────────────────────────
# direction_conflict = Y when fitted trend contradicts expected_direction
# + = higher value, higher Bad Rate = ascending WoE = ascending trend
# - = lower value, higher Bad Rate = descending trend

def check_direction_conflict(trend, expected_dir):
    ed = str(expected_dir).strip()
    if ed in ("?", "0", "", "nan"):
        return "N"  # no expectation or U-shape
    if trend is None:
        return "N"
    if ed == "+" and trend == "descending":
        return "Y"
    if ed == "-" and trend == "ascending":
        return "Y"
    return "N"

l1_df["direction_conflict"] = l1_df.apply(
    lambda r: check_direction_conflict(
        r["trend"],
        fdict_idx.loc[r["feature"], "expected_direction"] if r["feature"] in fdict_idx.index else "?"
    ), axis=1
)
print(f"Direction conflicts: {(l1_df['direction_conflict'] == 'Y').sum()}")

# ── IV rules (spec 6.2.2) ───────────────────────────────────────────
iv_min = L1["iv_min"]
iv_review_lo, iv_review_hi = L1["iv_review_band"]
iv_leak = L1["iv_leak_review"]

l1_df["iv_category"] = "NORMAL"
l1_df.loc[l1_df["iv_dev"] < iv_min, "iv_category"] = "LOW"
l1_df.loc[(l1_df["iv_dev"] >= iv_review_lo) & (l1_df["iv_dev"] < iv_review_hi), "iv_category"] = "REVIEW_BAND"
l1_df.loc[l1_df["iv_dev"] >= iv_leak, "iv_category"] = "LEAK_REVIEW"

print(f"\nIV categories:")
print(l1_df["iv_category"].value_counts())

# Features needing CV IV check: REVIEW_BAND and LEAK_REVIEW
needs_cv = l1_df[l1_df["iv_category"].isin(["REVIEW_BAND", "LEAK_REVIEW"])]["feature"].tolist()
print(f"\nFeatures needing CV IV: {len(needs_cv)}")

In [ ]:
# ── Cross-validated IV re-check (spec 6.2.3) ────────────────────

def compute_iv_from_table(tbl):
    """Compute IV from a binning table with +0.5 smoothing on empty counts."""
    # Exclude Totals row
    core = tbl[tbl.index != tbl.index[-1]].copy()
    if "Count" not in core.columns or "Event" not in core.columns:
        return 0.0
    counts = core["Count"].values.astype(float)
    events = core["Event"].values.astype(float)
    non_events = counts - events
    total_events = events.sum()
    total_non_events = non_events.sum()
    if total_events == 0 or total_non_events == 0:
        return 0.0
    # Add 0.5 for empty bins
    events_sm = np.where(events == 0, 0.5, events)
    non_events_sm = np.where(non_events == 0, 0.5, non_events)
    # Recalculate totals with smoothed values
    total_e_sm = events_sm.sum()
    total_ne_sm = non_events_sm.sum()
    dist_e = events_sm / total_e_sm
    dist_ne = non_events_sm / total_ne_sm
    iv = np.sum((dist_ne - dist_e) * np.log(dist_ne / dist_e))
    return iv

def cv_iv_for_feature(feat, X_full, y_full, strat, n_folds, dtype, special_codes, expected_dir):
    """Compute cross-validated IV: fit on 4 folds, IV on held-out fold."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_ivs = []
    is_cat = (dtype == "categorical")
    mono = BIN_CFG["monotonic_trend"]

    if str(expected_dir).strip() == "0":
        trends_to_try = ["peak", "valley"]
    else:
        trends_to_try = [mono]

    for train_idx, test_idx in skf.split(X_full, strat):
        x_train, y_train = X_full.iloc[train_idx], y_full.iloc[train_idx]
        x_test, y_test = X_full.iloc[test_idx], y_full.iloc[test_idx]

        best_iv_fold = -1
        for tr in trends_to_try:
            kwargs = dict(
                name=feat,
                dtype="categorical" if is_cat else "numerical",
                solver=BIN_CFG["solver"],
                monotonic_trend=tr if not is_cat else None,
                min_bin_size=BIN_CFG["min_bin_size"],
                min_bin_n_event=BIN_CFG["min_bin_n_event"],
                max_n_prebins=BIN_CFG["max_n_prebins"],
                max_n_bins=BIN_CFG["max_n_bins"],
            )
            if is_cat:
                kwargs["cat_cutoff"] = BIN_CFG["cat_cutoff"]
            if special_codes is not None:
                kwargs["special_codes"] = special_codes

            ob = OptimalBinning(**kwargs)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    ob.fit(x_train, y_train)
                except (ValueError, TypeError):
                    if is_cat and "cat_cutoff" in kwargs:
                        kw2 = {k: v for k, v in kwargs.items() if k != "cat_cutoff"}
                        ob = OptimalBinning(**kw2)
                        try:
                            ob.fit(x_train, y_train)
                        except Exception:
                            continue
                    else:
                        continue

            # Transform test data to bins, then compute IV on held-out
            try:
                tbl = ob.binning_table.build()
                # Apply bins to test data
                x_test_binned = ob.transform(x_test, metric="bins")
                # Build a cross-tab on test data
                test_df = pd.DataFrame({"bin": x_test_binned, "event": y_test.values})
                agg = test_df.groupby("bin").agg(Count=("event", "count"), Event=("event", "sum")).reset_index()
                agg.index = agg["bin"]
                iv_fold = compute_iv_from_table(pd.concat([agg, pd.DataFrame({"Count": [agg["Count"].sum()], "Event": [agg["Event"].sum()]}, index=["Totals"])]))
                if iv_fold > best_iv_fold:
                    best_iv_fold = iv_fold
            except Exception:
                best_iv_fold = 0.0

        fold_ivs.append(best_iv_fold)
    return round(np.mean(fold_ivs), 6)

# ── Compute CV IV for features that need it ───────────────────────
cv_iv_map = {}
for i, feat in enumerate(needs_cv):
    info = fdict_idx.loc[feat]
    dtype = str(info.get("dtype", "numerical")).strip()
    sc = parse_special_codes(info.get("special_codes", ""))
    ed = str(info.get("expected_direction", "?")).strip()
    x = dev_gb[feat].copy()
    if dtype == "categorical":
        x = x.astype(str).replace("nan", np.nan)
    else:
        x = pd.to_numeric(x, errors="coerce")
    iv_cv = cv_iv_for_feature(feat, x, y, strat_key, L1["cv_iv_folds"], dtype, sc, ed)
    cv_iv_map[feat] = iv_cv
    if (i + 1) % 20 == 0:
        print(f"  CV IV done: {i+1}/{len(needs_cv)}")

l1_df["iv_cv"] = l1_df["feature"].map(cv_iv_map)
print(f"\nCV IV computed for {len(cv_iv_map)} features")
if cv_iv_map:
    cv_series = pd.Series(cv_iv_map)
    print(f"CV IV summary:\n{cv_series.describe().round(4)}")

In [ ]:
# ── Apply IV drop rules (spec 6.2.2) ────────────────────────────────

l1_df["L1_pass"] = "Y"
l1_df["L1_reason"] = pd.NA
l1_df["leak_review"] = pd.NA
l1_df["direction_review"] = pd.NA

# Rule 1: IV < iv_min -> drop IV_LOW
mask_low = l1_df["iv_dev"] < iv_min
l1_df.loc[mask_low, "L1_pass"] = "N"
l1_df.loc[mask_low, "L1_reason"] = "IV_LOW"

# Rule 2: review band (iv_min <= IV < iv_review_hi) -> check iv_cv
mask_review = (l1_df["iv_category"] == "REVIEW_BAND") & (l1_df["L1_pass"] == "Y")
for idx in l1_df[mask_review].index:
    iv_cv = l1_df.loc[idx, "iv_cv"]
    if pd.notna(iv_cv) and iv_cv < iv_min:
        l1_df.loc[idx, "L1_pass"] = "N"
        l1_df.loc[idx, "L1_reason"] = "IV_CV_LOW"

# Rule 3: IV > iv_leak_review -> set leak_review = PENDING
mask_leak = (l1_df["iv_category"] == "LEAK_REVIEW") & (l1_df["L1_pass"] == "Y")
l1_df.loc[mask_leak, "leak_review"] = "PENDING"

# Rule 4: direction_conflict = Y -> needs review
mask_dir = (l1_df["direction_conflict"] == "Y") & (l1_df["L1_pass"] == "Y")
# direction_conflict features need review even if already passed other checks

# ── Human review via config/l1_review.csv ──────────────────────────
l1_review_path = os.path.join(PROJECT_ROOT, cfg["review_files"]["l1_review"])

review_needed = l1_df[
    ((l1_df["leak_review"] == "PENDING") | (l1_df["direction_conflict"] == "Y"))
    & (l1_df["L1_pass"] == "Y")
].copy()

if os.path.exists(l1_review_path):
    l1_review = pd.read_csv(l1_review_path)
    print(f"l1_review.csv loaded: {len(l1_review)} rows")
    review_idx = l1_review.set_index("feature")

    for idx in review_needed.index:
        feat = l1_df.loc[idx, "feature"]
        if feat in review_idx.index:
            row = review_idx.loc[feat]
            decision = str(row["decision"]).strip().upper()
            reason = str(row.get("reason", "")).strip()
            review_type = str(row.get("review_type", "")).strip().upper()

            if decision == "DROP":
                l1_df.loc[idx, "L1_pass"] = "N"
                if review_type == "LEAK":
                    l1_df.loc[idx, "L1_reason"] = "LEAKAGE"
                    l1_df.loc[idx, "leak_review"] = reason
                elif review_type == "DIRECTION":
                    l1_df.loc[idx, "L1_reason"] = "DIRECTION_CONFLICT"
                    l1_df.loc[idx, "direction_review"] = reason
                else:
                    l1_df.loc[idx, "L1_reason"] = review_type or "REVIEW_DROP"
            elif decision == "KEEP":
                if l1_df.loc[idx, "leak_review"] == "PENDING":
                    l1_df.loc[idx, "leak_review"] = reason
                if l1_df.loc[idx, "direction_conflict"] == "Y":
                    l1_df.loc[idx, "direction_review"] = reason
        else:
            # No row in review file -> PENDING
            l1_df.loc[idx, "L1_pass"] = "PENDING"
            l1_df.loc[idx, "L1_reason"] = "PENDING_REVIEW"
else:
    print(f"\u26a0 l1_review.csv not found: {l1_review_path}")
    if len(review_needed) > 0:
        for idx in review_needed.index:
            l1_df.loc[idx, "L1_pass"] = "PENDING"
            l1_df.loc[idx, "L1_reason"] = "PENDING_REVIEW"
        print(f"  {len(review_needed)} feature(s) set to PENDING")

n_pass = (l1_df["L1_pass"] == "Y").sum()
n_fail = (l1_df["L1_pass"] == "N").sum()
n_pending = (l1_df["L1_pass"] == "PENDING").sum()
print(f"\nL1 results: {n_pass} pass, {n_fail} dropped, {n_pending} pending")
print(f"\nDrop reasons:")
print(l1_df[l1_df["L1_pass"] == "N"]["L1_reason"].value_counts().to_string())
if n_pending > 0:
    print(f"\nPENDING features:")
    for _, r in l1_df[l1_df["L1_pass"] == "PENDING"].iterrows():
        print(f"  {r['feature']}: iv_dev={r['iv_dev']:.4f}, iv_cv={r.get('iv_cv', np.nan)}, "
              f"trend={r['trend']}, leak_review={r['leak_review']}, dir_conflict={r['direction_conflict']}")

In [ ]:
# ── Save binning objects to binning/L1/<feature>.pkl ───────────────
# Idempotent: remove .pkl files from earlier runs for features no longer in L1
# (e.g. newly dropped at L0 for COMPLIANCE), so the folder mirrors this run only.
stale_pkls = sorted(f for f in os.listdir(BINNING_DIR)
                    if f.endswith(".pkl") and f[:-4] not in binning_objects)
for f in stale_pkls:
    os.remove(os.path.join(BINNING_DIR, f))
if stale_pkls:
    print(f"Removed {len(stale_pkls)} stale binning object(s) from earlier runs: {stale_pkls}")

saved_ct = 0
for feat, ob in binning_objects.items():
    pkl_path = os.path.join(BINNING_DIR, f"{feat}.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(ob, f)
    saved_ct += 1
print(f"Saved {saved_ct} binning objects to {BINNING_DIR}")

# ── Save binning tables to reports/03_L1_binning.xlsx ──────────────
tmp_xl_bin = os.path.join(_TMP, "03_L1_binning.xlsx")
with pd.ExcelWriter(tmp_xl_bin, engine="openpyxl") as writer:
    for feat in sorted(binning_tables.keys()):
        tbl = binning_tables[feat].copy()
        tbl.insert(0, "feature", feat)
        sheet_name = feat[:31]  # Excel sheet name limit
        tbl.to_excel(writer, sheet_name=sheet_name, index=True)

    # Summary sheet
    l1_df.sort_values(["L1_pass", "iv_dev"], ascending=[False, False]).to_excel(
        writer, sheet_name="summary", index=False
    )
shutil.copyfile(tmp_xl_bin, os.path.join(REPORTS, "03_L1_binning.xlsx"))
print(f"Saved binning report: {REPORTS}/03_L1_binning.xlsx")

# ── Save review-needed report if any PENDING ─────────────────────
pending_features = l1_df[l1_df["L1_pass"] == "PENDING"]
if len(pending_features) > 0:
    review_xl = os.path.join(_TMP, "03_L1_review_needed.xlsx")
    with pd.ExcelWriter(review_xl, engine="openpyxl") as writer:
        cols = ["feature", "iv_dev", "iv_cv", "trend", "leak_review", "direction_conflict"]
        pending_features[cols].to_excel(writer, sheet_name="review_needed", index=False)
        # Add binning tables for each pending feature
        for _, r in pending_features.iterrows():
            feat = r["feature"]
            if feat in binning_tables:
                tbl = binning_tables[feat].copy()
                tbl.insert(0, "feature", feat)
                sheet_name = f"bin_{feat[:27]}"
                tbl.to_excel(writer, sheet_name=sheet_name, index=True)
    shutil.copyfile(review_xl, os.path.join(REPORTS, "03_L1_review_needed.xlsx"))
    print(f"Saved review-needed report: {REPORTS}/03_L1_review_needed.xlsx")

In [ ]:
# ── Update feature log ─────────────────────────────────────────────────
# L1 adds columns: iv_dev, iv_cv, n_bins, trend, direction_conflict,
# leak_review, L1_pass, L1_reason. Update dropped_at/drop_rule for L1 drops.

l1_cols = ["feature", "iv_dev", "iv_cv", "n_bins", "trend",
           "direction_conflict", "leak_review", "direction_review",
           "L1_pass", "L1_reason"]
l1_merge = l1_df[l1_cols].copy()  # Ensure 'L1_pass' and 'L1_reason' are included in l1_cols

# Drop any existing L1 columns from flog (idempotent re-run)
l1_only_cols = [c for c in l1_merge.columns if c != "feature"]
flog_clean = flog.drop(columns=[c for c in l1_only_cols if c in flog.columns])

# Merge L1 results into feature log
flog_updated = flog_clean.merge(l1_merge, on="feature", how="left")

# For features that didn't pass L0, L1 columns stay NaN
# Update dropped_at and drop_rule for L1 drops (only if not already dropped at L0)
mask_l1_drop = (flog_updated["L1_pass"] == "N") & (flog_updated["dropped_at"].isna())
flog_updated.loc[mask_l1_drop, "dropped_at"] = "L1"
flog_updated.loc[mask_l1_drop, "drop_rule"] = flog_updated.loc[mask_l1_drop, "L1_reason"]

# Also mark PENDING as dropped_at = L1 with PENDING_REVIEW
mask_l1_pending = (flog_updated["L1_pass"] == "PENDING") & (flog_updated["dropped_at"].isna())
flog_updated.loc[mask_l1_pending, "dropped_at"] = "L1"
flog_updated.loc[mask_l1_pending, "drop_rule"] = "PENDING_REVIEW"

# Ensure no void-typed columns (all-null columns default to void in Spark)
for c in flog_updated.columns:
    if flog_updated[c].isna().all():
        flog_updated[c] = flog_updated[c].astype(str)

# Save to Delta table
sdf = spark.createDataFrame(flog_updated)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL}  ({spark.table(FLOG_TBL).count()} rows)")

# Save CSV
csv_path = os.path.join(ARTIFACTS, "feature_log.csv")
flog_updated.to_csv(csv_path, index=False)
print(f"Saved CSV : {csv_path}")

# ── Report: reports/03_L1.xlsx ──────────────────────────────────────
tmp_xl = os.path.join(_TMP, "03_L1.xlsx")
with pd.ExcelWriter(tmp_xl, engine="openpyxl") as writer:
    meta = pd.DataFrame([{
        "module": "E5_L1", "run_time": datetime.now().isoformat(),
        "config_hash": common.config_hash(cfg),
        "funnel_config_hash": funnel_config_hash,
        "dev_rows_gb": len(dev_gb),
        "features_in": len(l1_df), "features_pass": int(n_pass),
        "features_drop": int(n_fail), "features_pending": int(n_pending),
    }])
    meta.to_excel(writer, sheet_name="meta", index=False)

    reason_cts = l1_df[l1_df["L1_pass"] == "N"].groupby("L1_reason").size().reset_index(name="count")
    reason_cts.to_excel(writer, sheet_name="counts_by_reason", index=False)

    l1_df.sort_values(["L1_pass", "iv_dev"], ascending=[False, False]).to_excel(
        writer, sheet_name="feature_detail", index=False
    )

shutil.copyfile(tmp_xl, os.path.join(REPORTS, "03_L1.xlsx"))
print(f"Saved report: {REPORTS}/03_L1.xlsx")

In [ ]:
# ── Acceptance Criteria (E5 — L1) ─────────────────────────────────
print("=" * 60)
print("ACCEPTANCE CRITERIA (E5 — L1)")
print("=" * 60)
checks = []

# 1. Regular bins meet min_bin_size, min_bin_n_event, max_n_bins
min_bs = BIN_CFG["min_bin_size"]
min_ne = BIN_CFG["min_bin_n_event"]
max_nb = BIN_CFG["max_n_bins"]
n_total = len(dev_gb)

l1_passed_feats = set(l1_df[l1_df["L1_pass"] == "Y"]["feature"])
bin_violations = []
for feat, tbl in binning_tables.items():
    if feat not in l1_passed_feats:
        continue  # only validate survivors
    core = tbl[~tbl.index.isin(["Special", "Missing", "Totals", tbl.index[-1]])]
    if len(core) == 0:
        continue
    # Check n_bins
    if len(core) > max_nb:
        bin_violations.append(f"{feat}: {len(core)} bins > max {max_nb}")
    # Check min_bin_size and min_bin_n_event
    if "Count" in core.columns and "Event" in core.columns:
        for bin_name, row in core.iterrows():
            ct = row["Count"]
            ev = row["Event"]
            share = ct / n_total
            if share < min_bs * 0.9:  # small tolerance for rounding
                bin_violations.append(f"{feat} bin '{bin_name}': size {share:.3f} < {min_bs}")
            if ev < min_ne * 0.9:
                bin_violations.append(f"{feat} bin '{bin_name}': events {ev} < {min_ne}")

# Bin constraints are soft in OptBinning; log as warning, don't hard-fail
if bin_violations:
    feats_with_violations = len(set(v.split(" bin ")[0].split(":")[0] for v in bin_violations))
    print(f"  ⚠ WARNING: {len(bin_violations)} bin constraint violations across "
          f"{feats_with_violations} features (OptBinning soft constraints)")
    for v in bin_violations[:10]:
        print(f"    {v}")
    checks.append((f"Bin constraints (soft): {len(bin_violations)} violations across {feats_with_violations} feats", True))
else:
    checks.append(("Regular bins meet constraints", True))

# 2. Every feature in review band or above iv_leak_review has iv_cv
mask_needs_cv = l1_df["iv_category"].isin(["REVIEW_BAND", "LEAK_REVIEW"])
cv_missing = l1_df[mask_needs_cv & l1_df["iv_cv"].isna()]
checks.append((f"Review/leak features have iv_cv ({len(cv_missing)} missing)",
               len(cv_missing) == 0))

# 3. No feature is PENDING
pending = l1_df[l1_df["L1_pass"] == "PENDING"]
checks.append((f"No feature is PENDING ({len(pending)} pending)",
               len(pending) == 0))
if len(pending) > 0:
    print("  PENDING features requiring l1_review.csv:")
    for _, r in pending.iterrows():
        print(f"    {r['feature']}: iv_dev={r['iv_dev']:.4f}, iv_cv={r.get('iv_cv', 'N/A')}, "
              f"trend={r['trend']}, leak={r['leak_review']}, dir={r['direction_conflict']}")

# 4. Dropped features have reasons
dropped = l1_df[l1_df["L1_pass"] == "N"]
if len(dropped) > 0:
    has_reason = dropped["L1_reason"].notna().all()
    checks.append(("Dropped features have reasons", has_reason))
else:
    checks.append(("Dropped features have reasons (none dropped)", True))

# 5. Feature log table row count matches
tbl_ct = spark.table(FLOG_TBL).count()
checks.append((f"Feature log table rows = {tbl_ct}", tbl_ct == len(flog_updated)))

# 6. Feature log CSV exported
checks.append(("feature_log.csv exported", os.path.exists(csv_path)))

# 7. Binning objects saved
pkl_on_disk = {f[:-4] for f in os.listdir(BINNING_DIR) if f.endswith(".pkl")}
pkl_missing = sorted(set(binning_objects) - pkl_on_disk)
pkl_stale = sorted(pkl_on_disk - set(binning_objects))
if pkl_stale:
    print(f"  \u26a0 WARNING: {len(pkl_stale)} stale .pkl file(s) in {BINNING_DIR} not from this run: {pkl_stale}")
checks.append((f"Binning objects saved ({len(binning_objects) - len(pkl_missing)}/{len(binning_objects)}, "
               f"{len(pkl_stale)} stale)", len(pkl_missing) == 0 and len(pkl_stale) == 0))

all_pass = True
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
    if not ok:
        all_pass = False

# ── Run summary ─────────────────────────────────────────────────
summary = {
    "module": "E5_L1",
    "run_time": datetime.now().isoformat(),
    "elapsed_s": round(time.time() - _t0, 1),
    "config_hash": common.config_hash(cfg),
    "funnel_config_hash": funnel_config_hash,
    "dev_rows_gb": len(dev_gb),
    "features_total": len(l1_df),
    "features_pass": int(n_pass),
    "features_drop": int(n_fail),
    "features_pending": int(n_pending),
    "drop_reasons": l1_df[l1_df["L1_pass"]=="N"]["L1_reason"].value_counts().to_dict(),
    "pass_fail": "PASS" if all_pass else "FAIL",
}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
with open(os.path.join(REPORTS, "run_summaries", "s03_L1.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary: {REPORTS}/run_summaries/s03_L1.json")

# Assert all pass
for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"

print("\n\u2705 All acceptance criteria passed.")